In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import matplotlib.colors as colors
# import holoviews as hv
from matplotlib.patches import Rectangle

from cycler import cycler
import matplotlib as mpl
import os

In [ ]:
from scipy.signal import savgol_filter
import gzip
import matplotlib as mpl
from cycler import cycler
# hv.extension('bokeh')
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.signal import peak_widths
from numpy.fft import rfft, rfftfreq

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 14})

In [ ]:
from scipy.integrate import quad

In [ ]:
# Gain table
q1 = 100.075
q2 = 100.103
q3 = 100.087
q4 = 100.082

s1 = 100.104
s2 = 99.977
s3 = 100.183

In [ ]:
def load1dFig4(i):
    return np.loadtxt(os.path.join('..', '20230512 HMIA13-5 QD', 'data', str(i), 'data.tsv'))

def load2dFig4(i, num):
    tmp = np.loadtxt(os.path.join('..', '20230512 HMIA13-5 QD', 'data', str(i), 'data.tsv'))
    numrows = np.floor(tmp.shape[0] / num)
    data_dict = {}
    for j in range(tmp.shape[1]):
        data_dict[str(j)] = tmp[:num*int(numrows), j].reshape((int(numrows), num)).transpose()
    return data_dict
# fast axis is column-wise

In [ ]:
from scipy.special import gamma as Gamma
# import matplotlib.pyplot as plt
from scipy.special import digamma, hyp2f1
from scipy.integrate import quad
from scipy.optimize import fsolve

#Following equations defined in PhysRevB.56.1848
def JsumComponent(t, R, Ec, T):
    #The infinite sum component of J(t) defined in above paper
    gamma = np.euler_gamma
    beta = 1.0/T
    x=beta*Ec/(2*R*np.pi**2)
    y=np.exp(-4*np.pi**2 * t/beta)
    return -(1/np.pi)*(2*gamma+digamma(-x)+digamma(x)+2*np.log(1-y)+(y/(1+x)) *hyp2f1(1,1+x,2+x,y) + (y/(1-x)) * hyp2f1(1, 1-x,2-x,y))

def J(t, R, Ec, T):
    #R=series resistance
    #assuming charge of electron, h = 1 (so Rk = e^2/h = 1)
    #kboltzmann = 1
    #Also assuming Ec = 1/2C (got from Ingold and Nazarov Chapter in Single Charge Tunneling Book, eq 66)
    wc = 2*Ec/R
    beta = 1.0/T

    return np.pi*R*((1-np.exp(-wc*abs(t)))*(np.tan(beta*wc/(4*np.pi))**(-1) -1j)- 4*np.pi*abs(t)/beta + JsumComponent(abs(t), R, Ec,T))

### EB is defined in F.D. Parmentier et al. Nature (2011)
@np.vectorize
def EB(V, R, Ec, T, shift = 0):
    #R = series resistnace in h/e^2
    #Ec = charging energy in ueV
    #T = temp in ueV
    # Rt shuold actually be Ginf in e^2/h
    #shift accounts for a shift in where zero bias is (offset of DAC channel etc)
    beta = 1.0/T
    V = V-shift
    def integrand(t, R, Ec, T):
        tmp = np.zeros((1))
        tmp.dtype = np.longdouble
        tmp = 4*(np.pi**3)*t/(beta**2)*np.imag(np.exp(J(t, R,Ec,T)))*np.cos(2*np.pi*V*t)*(np.sinh(2*np.pi**2 * t/beta)**-2)#.astype(np.longdouble)
        return tmp
    return 1*(2*quad(integrand,0, np.inf, args = (R, Ec,T))[0])

### EB is defined in F.D. Parmentier et al. Nature (2011)
@np.vectorize
def EBzero(R, Ec, T, shift = 0):
    #R = series resistnace in h/e^2
    #Ec = charging energy in ueV
    #T = temp in ueV
    # Rt shuold actually be Ginf in e^2/h
    #shift accounts for a shift in where zero bias is (offset of DAC channel etc)
    V = 0
    beta = 1.0/T
    V = V-shift
    def integrand(t, R, Ec, T):
        tmp = np.zeros((1))
        tmp.dtype = np.longdouble
        tmp = 4*(np.pi**3)*t/(beta**2)*np.imag(np.exp(J(t, R,Ec,T)))*np.cos(2*np.pi*V*t)*(np.sinh(2*np.pi**2 * t/beta)**-2)#.astype(np.longdouble)
        return tmp
    return 1*(2*quad(integrand,0, np.inf, args = (R, Ec,T))[0])


In [ ]:
def sineQ(Vp, A, w, phi, c, l):
    return A*np.sin(w*Vp+phi) + c + l*Vp

def KNtheory(tau0, T, Ec, Z, idcstart=-5e-9, idcstop=5e-9, npoints=51):
    idc = np.linspace(idcstart, idcstop, npoints)
    V = 1*idc*25813/3
    tauV = np.zeros((len(V),))
    EB0 = EB(0, Z, Ec, T)
    EBV = EB(V, Z, Ec, T)
    C = (tau0/(1-tau0))*(1/(EB0+1))
    tauV = (C*(EBV+1))/(C*(EBV+1) + 1)
    return tauV


def KNtheory2(tau0, T, Ec, Z):
    idc = np.linspace(-1.5e-8, 1.5e-8, 151)
    V = 1*idc[30:120]*25813/3
    tauV = np.zeros((len(V),))
    EB0 = EB(0, Z, Ec, T)
    EBV = EB(V, Z, Ec, T)
    C = (tau0/(1-tau0))*(1/(EB0+1))
    tauV = (C*(EBV+1))/(C*(EBV+1) + 1)
    return tauV

def KNtheoryfull(tau0, V, T, Ec, Z):
    tauV = np.zeros((len(V),))
    EB0 = EB(0, Z, Ec, T)
    EBV = EB(V, Z, Ec, T)
    C = (tau0/(1-tau0))*(1/(EB0+1))
    tauV = (C*(EBV+1))/(C*(EBV+1) + 1)
    return tauV

@np.vectorize
def FurusakiMatveev(tau1, tau2, EcbykT, deltaVgbyDelta):
    Gamma = (1-tau1) + (1-tau2) - 2*np.sqrt((1-tau1)*(1-tau2))*np.cos(2*np.pi*deltaVgbyDelta)
    def integrand(x, Gamma, EcbykT):
        gamma = np.exp(0.5772)
        return (Gamma/np.cosh(x))**2 / ((x*np.pi**2/(gamma*EcbykT))**2 + Gamma**2)
    
    
    return 0.5*(1 - quad(integrand,0, np.inf, args = (Gamma,EcbykT))[0])
@np.vectorize
def QFurusakiMatveev(tau1, tau2, EcbykT):
    Gmax = FurusakiMatveev(tau1, tau2, EcbykT, 0)
    Gmin = FurusakiMatveev(tau1, tau2, EcbykT, 1/2)

    return (Gmax - Gmin)/(Gmax + Gmin)

In [ ]:
dat = load2dFig4(796, 151)
idc = dat['2']
botqpc = dat['1']
vt = dat['5']
vr = dat['3']

G = 3*vt/(vt+vr)
tau = (1/G - 1)**-1
tauVfig5top = np.zeros(tau.shape)
start = 30
end = 52

start2 = start
start3 = 5

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
im = ax[0].pcolormesh(botqpc, 1e6*idc*25813/3, np.abs(tau), cmap='magma',vmin=1e-3, vmax=1)
cbar = fig.colorbar(im, ax=ax[0], extend='max')
ax[0].set_xlabel('V$_g$ (V)')
begin=5
end=-5
# tauV = KNtheory(tau, Z, Ec, T, V)
poptfig5top = np.zeros((3,151))
pcovfig5top = np.zeros((3,151))
# tauinf = np.zeros((idc.shape[1]))
Z = 1
# T = 2*1*4.3e-6
# Ec = 12*T
# V = np.linspace(-3*Ec, 3*Ec,3001)
reftau0top =  np.zeros((botqpc.shape[1]))
reftauinftop =  np.zeros((botqpc.shape[1]))
# tauV2 = KNtheory(tau, Z, Ec, T, V)
# start2 = 5
j=0
paramBounds = ([2e-6, 40e-6, 0.99999*Z], [8e-6, 100e-6, 1.000001*Z])
# paramBounds = ([2e-6, 90e-6, 0.99999*Z], [8e-6, 110e-6, 1.000001*Z])

for i in range(idc.shape[1]):
    tau0 = tau[75,i]
    reftau0top[i] = tau0
    if i>=10 and i<63:
#         print(botqpc[0,i])
#         ax[1].plot(0,0)
        ax[1].plot(1e6*(idc[begin:end,i] - idc[75,i])*25813/3, tau[begin:end,i],'k-', markersize=1, lw=1)
        poptfig5top[:,j], pcovifig5top = curve_fit(KNtheory, tau0, tau[50:101,i], p0=[4.3e-6, 90e-6, Z], bounds=paramBounds)
        pcovfig5top[:,j] = np.diag(pcovifig5top)
        print(poptfig5top[:,j])
        tauVfig5top[begin:end,i] = KNtheoryfull(tau0, (idc[begin:end,i] - 1*idc[75,i])*25813/3, poptfig5top[0,j], poptfig5top[1,j], poptfig5top[2,j])
        print(tauVfig5top[end-1,i])
        reftauinftop[i] = tauVfig5top[end-1,i]
        ax[1].plot(1e6*(idc[begin:end,i] - 1*idc[75,i])*25813/3, tauVfig5top[begin:end,i], ls='--', color='b', lw=0.75)
        j=j+1
#         print(popt5)
# #         ax[1].plot(1e6*vsdc[start:end,i]*25813/3, Gfull(1e6*vsdc[start:end,i]*25813/3, 1, 55, 4, 0.31, 4), 'r--')
#         popt5 = test(1e6*vsdc[start:end,i]*25813/3, tau[start:end, i],  1, 55, 4, 0.08, 0)
#         ax[1].plot(1e6*vsdc[start:end,i]*25813/3, Gfull(1e6*vsdc[start:end,i]*25813/3, popt5[0], popt5[1], popt5[2], popt5[3], popt5[4]), 'r--')



# ax[1].axvline(1e6*idc[41,0]*25813/3)
# ax[1].set_xlim(-100, 100)
ax[0].set_ylabel('V$_{dc}$  ($\mu$V)')
ax[1].set_xlabel('V$_{dc}$  ($\mu$V)')
cbar.ax.set_ylabel('$\u03C4_{top}$')
ax[1].set_ylabel('$\u03C4_{top}$')
ax[1].grid(ls='--', lw=0.4)
# ax[1].set_ylim(0, 1.1)
fig.tight_layout()

# fig.savefig('figures/DCBtopQPConeRk.jpeg', dpi=300)
# fig.savefig('figures/DCBfitsKNmodel_oldZ1.jpeg', dpi=600)

In [ ]:
dat = load2dFig4(802, 151)
idc = dat['2']
botqpc = dat['1']
vt = dat['5']
vr = dat['3']

G = 3*vt/(vt+vr)
tau = (1/G - 1)**-1
tauVfig5bot = np.zeros(tau.shape)
start = 30
end = 52

start2 = start
start3 = 5

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
im = ax[0].pcolormesh(botqpc, 1e6*idc*25813/3, np.abs(tau), cmap='magma',vmin=1e-3, vmax=1)
cbar = fig.colorbar(im, ax=ax[0], extend='max')
ax[0].set_xlabel('V$_g$ (V)')
begin=5
end=-5
# tauV = KNtheory(tau, Z, Ec, T, V)
poptfig5bot = np.zeros((3,151))
pcovfig5bot = np.zeros((3,151))
# tauinf = np.zeros((idc.shape[1]))
Z = 1
# T = 2*1*4.3e-6
# Ec = 12*T
# V = np.linspace(-3*Ec, 3*Ec,3001)
reftau0bot =  np.zeros((botqpc.shape[1]))
reftauinfbot =  np.zeros((botqpc.shape[1]))
# tauV2 = KNtheory(tau, Z, Ec, T, V)
# start2 = 5
j=0
paramBounds = ([2e-6, 40e-6, 0.99999*Z], [8e-6, 100e-6, 1.000001*Z])
# paramBounds = ([2e-6, 90e-6, 0.99999*Z], [8e-6, 110e-6, 1.000001*Z])

for i in range(idc.shape[1]):
    tau0 = tau[75,i]
    reftau0bot[i] = tau0
    if i>=20 and i<63:
#         print(botqpc[0,i])
#         ax[1].plot(0,0)
        ax[1].plot(1e6*(idc[begin:end,i] - idc[75,i])*25813/3, tau[begin:end,i],'k-', markersize=1, lw=1)
        poptfig5bot[:,j], pcovifig5bot = curve_fit(KNtheory, tau0, tau[50:101,i], p0=[4.3e-6, 95e-6, Z], bounds=paramBounds)
        pcovfig5bot[:,j] = np.diag(pcovifig5bot)
        print(poptfig5bot[:,j])
        tauVfig5bot[begin:end,i] = KNtheoryfull(tau0, (idc[begin:end,i] - 1*idc[75,i])*25813/3, poptfig5bot[0,j], poptfig5bot[1,j], poptfig5bot[2,j])
        print(tauVfig5bot[end-1,i])
        reftauinfbot[i] = tauVfig5bot[end-1,i]
        ax[1].plot(1e6*(idc[begin:end,i] - 1*idc[75,i])*25813/3, tauVfig5bot[begin:end,i], ls='--', color='b', lw=0.75)
        j=j+1
#         print(popt5)
# #         ax[1].plot(1e6*vsdc[start:end,i]*25813/3, Gfull(1e6*vsdc[start:end,i]*25813/3, 1, 55, 4, 0.31, 4), 'r--')
#         popt5 = test(1e6*vsdc[start:end,i]*25813/3, tau[start:end, i],  1, 55, 4, 0.08, 0)
#         ax[1].plot(1e6*vsdc[start:end,i]*25813/3, Gfull(1e6*vsdc[start:end,i]*25813/3, popt5[0], popt5[1], popt5[2], popt5[3], popt5[4]), 'r--')



# ax[1].axvline(1e6*idc[41,0]*25813/3)
# ax[1].set_xlim(-100, 100)
ax[0].set_ylabel('V$_{dc}$  ($\mu$V)')
ax[1].set_xlabel('V$_{dc}$  ($\mu$V)')
cbar.ax.set_ylabel('$\u03C4_{top}$')
ax[1].set_ylabel('$\u03C4_{top}$')
ax[1].grid(ls='--', lw=0.4)
# ax[1].set_ylim(0, 1.1)
fig.tight_layout()

# fig.savefig('figures/DCBtopQPConeRk.jpeg', dpi=300)
# fig.savefig('figures/DCBfitsKNmodel_oldZ1.jpeg', dpi=600)

In [ ]:
from matplotlib.gridspec import GridSpec
# figinset, inset_ax = plt.subplots()
fig4 = plt.figure(figsize=(8, 8),constrained_layout=True)
gs = fig4.add_gridspec(4, 3, width_ratios=(1,1,1))
# gs = GridSpec(3, 3, figure=fig1)
f4_ax1 = fig4.add_subplot(gs[0:2, :2])
f4_ax11 = fig4.add_subplot(gs[2:4, :2])
# f4_ax11 = fig4.add_subplot(gs[2, 0])
# f3_ax1.set_title('gs[0, :3]')
f4_ax2 = fig4.add_subplot(gs[0, 2])
f4_ax3 = fig4.add_subplot(gs[1, 2])
f4_ax4 = fig4.add_subplot(gs[2, 2])
f4_ax5 = fig4.add_subplot(gs[3, 2])
# f4_ax6 = fig4.add_subplot(gs[2:, :2])


f4_ax1.text(-0.1, 1, "(a)", fontsize=14, va="bottom", ha="right", transform=f4_ax1.transAxes)
f4_ax11.text(-0.1, 1, "(b)", fontsize=14, va="bottom", ha="right", transform=f4_ax11.transAxes)
f4_ax2.text(-0.1, 1, "(c)", fontsize=14, va="bottom", ha="right", transform=f4_ax2.transAxes)
f4_ax3.text(-0.1, 1, "(d)", fontsize=14, va="bottom", ha="right", transform=f4_ax3.transAxes)
f4_ax4.text(-0.1, 1, "(e)", fontsize=14, va="bottom", ha="right", transform=f4_ax4.transAxes)
f4_ax5.text(-0.1, 1, "(f)", fontsize=14, va="bottom", ha="right", transform=f4_ax5.transAxes)



sym = ['o', 's', 'v', 'd', 'p', '*', '<', 'o', 's', 'v', 'd', 'p', '*', '<','o', 's', 'v', 'd', 'p', '*', '<', 'o', 's', 'v', 'd', 'p', '*', '<']
cc = ['b','pink','k', 'r', 'olive', 'violet',  'c', 'gray', 'm','b','pink','k', 'r', 'olive', 'violet',  'c', 'gray', 'm']
botqpcarray = np.linspace(-2.92, -3.20, 29)
tau0botarray = np.zeros(botqpcarray.shape)


# fig, ax = plt.subplots(1, 2, figsize=(8,4))
ax0 = f4_ax1
ax1 = f4_ax11
# ax1 = f4_ax6
# ax1 = inset_axes(f4_ax1,
                    # width="30%", # width = 30% of parent_bbox
                    # height="30%", # height : 1 inch
                    # bbox_to_anchor=(0.1,0.35,1,1), bbox_transform=f4_ax1.transAxes,
                    # loc=3)
dat = load2dFig4(755, 86)
plunger = dat['2'][:, :]
topqpc = dat['1'][:, :]
vr = dat['3'][:, :]*100/q2
vt = dat['5'][:, :]*100/s1

#     tauinftop = tauinf(tau0top[240:301])
G = 3*vt/(vt+vr)
# tau0top = (1/G[0,:] - 1)**-1

# figtest, axtest = plt.subplots()
# figtest2, axtest2 = plt.subplots()
tau0top = np.zeros((G.shape[1]))
tau0toperr = np.zeros((G.shape[1]))
tauinftoperr = np.zeros((G.shape[1]))
tauinftop = np.zeros((G.shape[1]))
# for j,d in enumerate([ 217, 219, 221, 223]):
# for j,d in enumerate([590,595,600, 605, 610]):#enumerate(np.arange(498, 509, 1)):
# for j,d in enumerate([586, 588, 590, 592, 595, 596, 614]):#enumerate(np.arange(498, 509, 1)):
for j,d in enumerate([756, 757, 758, 759, 760, 761, 762, 764, 766]):#enumerate([755]):
    # begin=66
    botqpc = botqpcarray[d-755]
    print(botqpc)
    dat = load2dFig4(d, 86)
    plunger = dat['2'][:, :]
    topqpc = dat['1'][:, :]
    vr = dat['3'][:, :]*100/q2
    vt = dat['5'][:, :]*100/s1

#     tauinftop = tauinf(tau0top[240:301])
    mpl.style.use('default')
    G = 3*vt/(vt+vr)
    tau0bot = np.mean((1/(G[:,0]) - 1/1)**-1)
    tau0boterr = np.std(((1/(G[:,0]) - 1/1)**-1))#/np.sqrt(G.shape[0])
    print(f'Standard error of tau bottom = {tau0boterr}')
    if tau0bot>=1:
        tau0bot=1-5e-3
    tau0botarray[j] = tau0bot
    tauinfbot = np.interp(tau0bot, reftau0bot[20:55][np.argsort(reftau0bot[20:55])], reftauinfbot[20:55][np.argsort(reftau0bot[20:55])], left = 0, right = 1)
    slopeinterp = np.interp(tau0bot, reftau0bot[20:55][np.argsort(reftau0bot[20:55])], np.gradient(reftauinfbot[20:55][np.argsort(reftau0bot[20:55])])/np.gradient(reftau0bot[20:55][np.argsort(reftau0bot[20:55])]), left = 0, right = 1)
    tauinfboterr = slopeinterp*tau0boterr
    # axtest.plot(plunger[:,0], G[:,0], label=f'tau0 bottom = {tau0bot}, tauinf bottom = {tauinfbot}', color=cc[j])

    if j==0:
        for k in range(G.shape[1]):
            tau0top[k] = np.mean((1/(G[:,k]) - 1/1)**-1)
            tau0toperr[k] = np.std((1/(G[:,k]) - 1/1)**-1)
            tauinftop[k] = np.interp(tau0top[k], reftau0top[10:55][np.argsort(reftau0top[10:55])], reftauinftop[10:55][np.argsort(reftau0top[10:55])], left = 0, right = 1)
            slopeinterp = np.interp(tau0top[k], reftau0top[10:55][np.argsort(reftau0top[10:55])], np.gradient(reftauinftop[10:55][np.argsort(reftau0top[10:55])])/np.gradient(reftau0top[10:55][np.argsort(reftau0top[10:55])]), left = 0, right = 1)
            tauinftoperr[k] = slopeinterp*tau0toperr[k]
            # axtest2.plot(k, tau0top, 'rx')
            # axtest2.plot(k, tauinftop[k], 'bo')
            
    G0 = (1/tauinftop + 1/tauinfbot)**-1
    
    # tauinfbot = 
    # tauinfbot = tauinf(tau0bot)
    print(j)
    print('$\u03C4_{bottom} = $ %1.4f' %(tauinfbot))
    print('--')
    # tau0top = tautop1d
    # tauinftop = tauinf(tau0top)
    Q1 = np.zeros((G.shape[1])) # sinusoidal fit
    Q1err = np.zeros((G.shape[1])) # errors from sinusoidal fit
    Q2 = np.zeros((G.shape[1])) # max - min / max + min
    Q2err = np.zeros((G.shape[1])) # sem from min max 
    Q = np.zeros((G.shape[1]))
    Qerr = np.zeros((G.shape[1]))
    Gset = np.zeros((G.shape[1]))
    start=2
    Q2slice = np.zeros((3))
    # tauinftop = tauVfig5top[-6, 20:61]#tauinftop_prechargeQ[10:71]#tauinf(ftautop1dup)
    # tauinftop = tauinftop_postchargeQ[0:61]#tauinf(ftautop1dup)

    for i in range(G.shape[1]):
        if i>=0 and i < 51:
            Gsmooth = savgol_filter(G[start:,i], 11, 3, deriv=0)
            if j==6 and i>10:
                peaks, _ = find_peaks(Gsmooth, prominence=0.02)
                # for k in range(G.shape[1]):
                # axtest.plot(plunger[start:,i], Gsmooth[:]+0.1*i, 'k')
                # axtest.plot(plunger[start+peaks[1],i],Gsmooth[peaks[1]]+0.1*i,'r*')
                Gset[i] = Gsmooth[peaks[1]]

            Gmax = np.max(Gsmooth)
            Gmaxslice = np.max(Gsmooth[:30])
            Gmin = np.min(np.abs(Gsmooth))
            Q2[i] = (Gmax - Gmin)/(Gmax+Gmin)
            c = np.mean(Gsmooth)
            # Gset[i] = Gmaxslice
            A = Gmax - c
            w = 2*np.pi/6e-3 # 6 mV is the Coulomb peak periodicity
            phi = np.pi/2
            paramBounds = ([0, 2*np.pi/10e-3, -np.pi, 0.999*c-1e-4, -np.inf], [Gmax-c,2*np.pi/5e-3,np.pi, 1.001*c+1e-4, np.inf])
#             print(c)
            popt, pcov = curve_fit(sineQ, plunger[start:, i], Gsmooth, p0 = [A, w, phi, c, 0], bounds=paramBounds)
#             print(2*np.pi*popt[1]**-1)
            Q1[i] = popt[0]/popt[3]
            perr = np.sqrt(np.diag(pcov))
            Q1err[i] = Q1[i]*np.sqrt((perr[0]/popt[0])**2 + (perr[3]/popt[3])**2)
            '''plots for checking sinusoidal fits'''
 
            if tauinftop[i] > 0.996 or tauinfbot > 0.996:
                Q[i] = Q1[i]
                Qerr[i] = Q1err[i]
            else:
                Q[i] = Q2[i]

    '''plots for checking Q and tau vs top QPC gate'''
    skip=25
    if d<600:
        end = skip+10
    else:
        end = 100 
    # end=100 # 599
    
    ########### Q vs Vg #############
#     ax[1].axvline(topqpc[0,end], ls='--', color='k')
#     ax[1].plot(topqpc[0,end:], Q[end:], sym[2], color=cc[j], markersize=2)#, label='$\u03C4_{bot} = %1.3f$' %(tauinfbot))
# #     ax.plot(tautop1d[end:], Q[end:126], sym[2], label='$\u03C4_{bot} = %1.3f$' %(tauinfbot))
#     ax[1].plot(topqpc[0, skip:end], Q2[skip:end], sym[1],  color=cc[j], label='$Vg_{bot} = %1.3f$' %(botqpcarray[d-582]), markersize=2)
    #################################
    ########### Q vs tau inf (estimated) #############
#     ax[1].axvline(topqpc[0,end], ls='--', color='k')
    # ax1.plot(tauinf(tautop1dup)[end:], Q1[end:], sym[2], ls='-', color=cc[j], markersize=2)#, label='$\u03C4_{bot} = %1.3f$' %(tauinfbot))
#     ax.plot(tautop1d[end:], Q[end:126], sym[2], label='$\u03C4_{bot} = %1.3f$' %(tauinfbot))

    def asymScaling(tau, A):
        return A*np.sqrt(1 - tau)
    
    if j>=1 and j<7:
        # print(j)
    #     ax0.loglog(1-tauinftop, Q, sym[j], color=cc[j], label='$\u03C4_{bot} = %1.3f$' %(tauinfbot), markersize=2)
        if j==6 or j==5 or j==4 or j==3 or j==2:
            popt, pcov = curve_fit(asymScaling, tauinftop[10:24], Q[10:24], p0 = 0.57*22*np.sqrt((1-tau0bot)))
            # ax1.loglog(1-tauinftop[5:41], 0.57*22*np.sqrt((1-tauinftop[5:41]))*np.sqrt((1-tau0bot)), ls='--',color=cc[j], lw=0.75, label='${\u03C4}_1 = %1.4f$' %(tauinfbot))
            ax1.loglog(1-tauinftop[10:41], popt[0]*np.sqrt((1-tauinftop[10:41])), ls='--',color=cc[j], lw=0.75, label='${\u03C4}_1 = %1.3f$ $\pm$ %1.3f' %(tauinfbot, tauinfboterr))
            ax1.errorbar(1-tauinftop[10:51], Q[10:51], xerr = np.abs(tauinftoperr[10:51]), fmt = sym[j], color=cc[j])#, label='$\u03C4_{bot} = %1.3f$' %(tauinfbot), markersize=5)
            if j==3 or j==2:
                # popt, pcov = curve_fit(QFurusakiMatveev, tauinftop[10:24], Q[10:24], p0 = [tau0bot, 22])
                # print(f'Furusaki Matveev fit Ec/kT = {popt[1]}')
                taubotfit = 1 - (popt[0]/(0.57*22))**2
                ax1.loglog(1-tauinftop[10:24], QFurusakiMatveev(tauinftop[10:24], taubotfit, 22), color=cc[j])
    # tau0top = (1/G[0,:] - 1)**-1
    # ax0.plot(topqpc[0,:], tau0top)
        ax0.plot(tauinftop[:51], Q[:51], sym[j], color=cc[j], label='$\u03C4_{1} = %1.4f$' %(tau0bot))
        # ax0.legend(ncol=3)
        ax0.annotate('', xytext=(0.51, 0.9), xy=(0.51, 0.2),
            arrowprops=dict(arrowstyle="->"))
        ax0.annotate("$\u03C4_{1} \in [0.258, 0.999]$", xy=(0.55, 0.5))
        ax1.annotate('', xytext=(0.03, 0.9), xy=(0.03, 0.04),
            arrowprops=dict(arrowstyle="->"))
        ax1.annotate("$\u03C4_{1} \in [0.258, 0.999]$", xy=(0.035, 0.05))
        # axtest2.plot(topqpc[0,:51], Gset[:51], 'x-', color=cc[j])
        print(tauinftop[np.argmax(Gset/G0)])
    #################################
    # axtest2.set_xlim(0.995,1)
    # ax1.legend()
#     ax.plot(tautop1d[begin:end], Q2[begin:end],ls='--', label='$\u03C4_{bot} = %1.3f$' %(tauinfbot))
#     ax.plot(plunger[:,0], G[:,120], label='$\u03C4_{bot}^{\infty} = %1.3f$' %(tauinfbot))
# # #     ax.axvline(topqpc[0, 45])
#     ax2.plot(topqpc[0, :end], tauitop[65, 65:65+end], 'h', markersize=2, color='r')
#     ax2.plot(topqpc[0, :end], tauV[3000, 65:65+end], '^', markersize=2, color='k')
    '''plots for Q vs 1-tau_top'''
#     ax.plot(botqpc, tau0bot, 'b*')
#     ax.plot(botqpc, tauinfbot, 'k*')
    
#     else:
#         ax.plot(tauinftop[:10], Q[:10], sym[j], color=cc[j], label='$\u03C4_{bot} = %1.5f$' %(tauinfbot))
#         ax.plot(tauinftop[10:], Q2[10:], sym[j], color=cc[j])#, label='$\u03C4_{bot} = %1.5f$' %(tauinfbot))


# #     ax.plot(topqpc[0, :40], tauitop[65,tstart:tstart+np.minimum(tauitop.shape[1]-tstart, len(Q))-46], ls='--', label='$\u03C4_{bot} = %1.3f$' %(tauibot[1]))
# #     ax.axvline(1-tauitop[65,tstart+45])
# #     ax.loglog(1-tauitop[65,tstart:tstart+np.minimum(tauitop.shape[1]-tstart, len(Q))], Q[:np.minimum(tauitop.shape[1]-tstart, len(Q))],sym[j], color='b')#, label='$\u03C4_{bot} = %1.3f$' %(tauibot[j]))
# #     ax.loglog(1-tauV[3000,tstart:tstart+np.minimum(tauitop.shape[1]-tstart, len(Q))], Q[:np.minimum(tauitop.shape[1]-tstart, len(Q))],sym[j], color='k')#, label='$\u03C4_{bot} = %1.3f$' %(tauibot[j]))
# #
# #     ax.loglog(1-tauD, Qtheory, '-',color='red')
#     ax.loglog(1-tauinftop, 0.57*25*np.sqrt(1-tauinftop)*np.sqrt(1-tauinfbot), ls='--',color=cc[j])#, label='$\u03C4_{bot} = %1.5f$' %(tauinfbot))
# ax.set_xlabel('V$_p$ (V)')
# ax0.plot(botqpcarray, tau0botarray)
ax1.set_xlabel('1 - ${{\u03C4}}_{2}$')
ax0.set_xlabel('$\u03C4_{2}$')

# ax0.legend(loc='lower left')
# ax1.legend()
# ax.set_xlim(5e-3, 1)
# ax.set_ylim(5e-2, 1.1)
# ax.set_ylabel('Q (visibility)')
# ax.axhline(0, color='k')
# ax.set_ylabel('bottom QPC  (V)')
# cbar.ax.set_ylabel('G ($e^2/h$)')
ax1.set_ylabel('$\\tilde{\u03C4}_{2}$')
ax1.set_ylabel('Q')
ax0.set_ylabel('Q')
# ax0.axvline(0.99)
ax1.grid(ls='--', lw=0.4)
ax0.grid(ls='-.', lw=0.4)
# ax1.set_ylim(1e-4,1.1)
ax0.set_ylim(1e-4,1.1)
ax1.set_ylim(1e-2,1.1)
# ax0.set_xlim(-3.05,-2.8)

ax1.set_xlim(1e-3, 1)
# ax1.set_ylim(4e-2, 2)
# ax1.set_xlim(1e-3,1)
# ax1.axhline(5e-2, color='gray', ls='--')
# axtest.legend()


dat = load2dFig4(762, 86)
plunger = dat['2'][:, :]
topqpc = dat['1'][:, :]
vr = dat['3'][:, :]*100/q2
vt = dat['5'][:, :]*100/s1

#     tauinftop = tauinf(tau0top[240:301])
mpl.style.use('default')
G = 3*vt/(vt+vr)

#  G = 3*vt/(vt+vr)
tau0bot = np.mean((1/(G[:,0]) - 1/1)**-1)
tau0boterr = np.std((1/(G[:,0]) - 1/1)**-1)
tauinfbot = np.interp(tau0bot, reftau0bot[20:55][np.argsort(reftau0bot[20:55])], reftauinfbot[20:55][np.argsort(reftau0bot[20:55])], left = 0, right = 1)
slopeinterp = np.interp(tau0bot, reftau0bot[20:55][np.argsort(reftau0bot[20:55])], np.gradient(reftauinfbot[20:55][np.argsort(reftau0bot[20:55])])/np.gradient(reftau0bot[20:55][np.argsort(reftau0bot[20:55])]), left = 0, right = 1)
tauinfboterr = slopeinterp*tau0boterr

f4_ax2.set_title('$\u03C4_1 = %1.3f$ $\pm$ %1.3f' %(tauinfbot, tauinfboterr))
f4_ax2.plot(1000*plunger[:,0], G[:,0], 'b', label='$\u03C4_{2} = %1.3f$ $\pm$ %1.3f' %(tauinftop[0], tauinftoperr[0]))
f4_ax3.plot(1000*plunger[:,0], G[:,22], 'b', label='$\u03C4_{2} = %1.3f$ $\pm$ %1.3f' %(tauinftop[22], tauinftoperr[22]))
f4_ax4.plot(1000*plunger[:,0], G[:,37], 'b', label='$\u03C4_{2} = %1.3f$ $\pm$ %1.3f' %(tauinftop[37], tauinftoperr[37]))
f4_ax5.plot(1000*plunger[:,0], G[:,55], 'b', label='$\u03C4_{2} = %1.3f$ $\pm$ %1.3f' %(tauinftop[55], np.abs(tauinftoperr[55])))

f4_ax2.set_ylim(0,0.5)
f4_ax3.set_ylim(0,0.5)
f4_ax4.set_ylim(0,0.5)
f4_ax5.set_ylim(0,0.5)

f4_ax2.grid(ls='--', lw=0.4)
f4_ax3.grid(ls='--', lw=0.4)
f4_ax4.grid(ls='--', lw=0.4)
f4_ax5.grid(ls='--', lw=0.4)

f4_ax2.set_ylabel('$G (e^2/h)$')
f4_ax3.set_ylabel('$G (e^2/h)$')
f4_ax4.set_ylabel('$G (e^2/h)$')
f4_ax5.set_ylabel('$G (e^2/h)$')

f4_ax5.set_xlabel('$V_{pR}$ (mV)')


f4_ax2.set_xticklabels([])
f4_ax3.set_xticklabels([])
f4_ax4.set_xticklabels([])

f4_ax2.legend(framealpha=0.5)
f4_ax3.legend(framealpha=0.5)
f4_ax4.legend(framealpha=0.5)
f4_ax5.legend(framealpha=0.5)

# axtest2.set_ylim(0.9, 1)
fig4.set_constrained_layout(True)
# fig4.savefig('/Users/praveen/Library/CloudStorage/GoogleDrive-prvn@stanford.edu/Shared drives/GGG GDrive/QDots-2/Papers/Hybrid Dot APL/figure4_Dec2025.pdf', dpi=300)
# fig.tight_layout()
# fig.savefig('figures/Cooldown2_Goscillations.pdf', dpi=300)


In [ ]:
fig5 = plt.figure(figsize=(12, 8),constrained_layout=True)
gs = fig5.add_gridspec(2, 2, width_ratios=(1,1))
# gs = GridSpec(3, 3, figure=fig1)
f5_ax1 = fig5.add_subplot(gs[:1, 0])
f5_ax2 = fig5.add_subplot(gs[:1, 1])
f5_ax3 = fig5.add_subplot(gs[1:2, 0])
f5_ax4 = fig5.add_subplot(gs[1:2, 1])

# inset_ax = fig3.add_subplot(gs[:1, 2])
# f1_ax2cbar = fig1.add_subplot(gs[0, 3])
# plt.setp(f4_ax2.get_yticklabels(), visible=False)
# f3_ax2.set_title('gs[0, 3:]')

f5_ax1.text(-0.1, 1, "(a)", fontsize=14, va="bottom", ha="right", transform=f5_ax1.transAxes)
f5_ax2.text(-0.1, 1, "(b)", fontsize=14, va="bottom", ha="right", transform=f5_ax2.transAxes)
# f5_ax1.set_axis_off()
f5_ax3.text(-0.1, 1, "(c)", fontsize=14, va="bottom", ha="right", transform=f5_ax3.transAxes)
f5_ax4.text(-0.1, 1, "(d)", fontsize=14, va="bottom", ha="right", transform=f5_ax4.transAxes)

###################
dat = load2dFig4(802, 151)
idc = dat['2']
botqpc = dat['1']
vt = dat['5']
vr = dat['3']

G = 3*vt/(vt+vr)
tau = (1/G - 1)**-1
tauVfig5bot = np.zeros(tau.shape)
# start = 30
# end = 52

# start2 = start
# start3 = 5

# im = f5_ax1.pcolormesh(botqpc, 1e6*idc*25813/3, np.abs(tau), cmap='magma',vmin=1e-3, vmax=1)
# cbar = fig.colorbar(im, f5_ax1, extend='max')
# f5_ax1.set_xlabel('V$_g$ (V)')
begin=5
end=-5
# tauV = KNtheory(tau, Z, Ec, T, V)
poptfig5bot = np.zeros((3,151))
pcovfig5bot = np.zeros((3,151))
# tauinf = np.zeros((idc.shape[1]))
Z = 1
# T = 2*1*4.3e-6
# Ec = 12*T
# V = np.linspace(-3*Ec, 3*Ec,3001)
reftau0bot =  np.zeros((botqpc.shape[1]))
reftauinfbot =  np.zeros((botqpc.shape[1]))
# tauV2 = KNtheory(tau, Z, Ec, T, V)
# start2 = 5
j=0
paramBounds = ([1e-6, 90e-6, 0.99999*Z], [8e-6, 110e-6, 1.000001*Z])
for i in range(idc.shape[1]):
    tau0 = tau[75,i]
    reftau0bot[i] = tau0
    if i>=30 and i<51:
        j = i - 10
#         print(botqpc[0,i])
#         ax[1].plot(0,0)
        f5_ax1.plot(1e6*(idc[begin:end,i] - idc[76,i])*25813/3, tau[begin:end,i],'k-', markersize=1, lw=1)
        poptfig5bot[:,j], pcovifig5bot = curve_fit(KNtheory, tau0, tau[50:101,i], p0=[4.3e-6, 100e-6, Z], bounds=paramBounds)
        pcovfig5bot[:,j] = np.diag(pcovifig5bot)
        print(poptfig5bot[:,j])
        tauVfig5bot[begin:end,i] = KNtheoryfull(tau0, (idc[begin:end,i] - 1*idc[75,i])*25813/3, poptfig5bot[0,j], poptfig5bot[1,j], poptfig5bot[2,j])
        # print(tauVfig5bot[end-1,i])
        reftauinfbot[i] = tauVfig5bot[end-1,i]
        f5_ax1.plot(1e6*(idc[begin:end,i] - 1*idc[75,i])*25813/3, tauVfig5bot[begin:end,i], ls='--', color='m', lw=0.75)
        j=j+1
#         print(popt5)
# #         ax[1].plot(1e6*vsdc[start:end,i]*25813/3, Gfull(1e6*vsdc[start:end,i]*25813/3, 1, 55, 4, 0.31, 4), 'r--')
#         popt5 = test(1e6*vsdc[start:end,i]*25813/3, tau[start:end, i],  1, 55, 4, 0.08, 0)
#         ax[1].plot(1e6*vsdc[start:end,i]*25813/3, Gfull(1e6*vsdc[start:end,i]*25813/3, popt5[0], popt5[1], popt5[2], popt5[3], popt5[4]), 'r--')



# ax[1].axvline(1e6*idc[41,0]*25813/3)
# ax[1].set_xlim(-100, 100)
f5_ax1.set_ylabel('V$_{dc}$  ($\mu$V)')
f5_ax1.set_xlabel('V$_{dc}$  ($\mu$V)')
f5_ax1.set_ylabel('$\u03C4_{1}$')
f5_ax1.grid(ls='--', lw=0.4)
f5_ax1.axvline(idc[50,0]*25813/3*1e6, color='gray', ls='--')
f5_ax1.axvline(idc[101,0]*25813/3*1e6, color='gray', ls='--')

##############

start = 30
stop = 50

f5_ax2.plot(botqpc[0,start:stop], reftau0bot[start:stop], 'k', label='$V_{dc} = %1.1f \mu V$' %(idc[75,0]*25813*1e6/3))
f5_ax2.plot(botqpc[0,start:stop], tau[-6,start:stop], 'b', label='$V_{dc} = %1.1f \mu V$' %(idc[-6,0]*25813*1e6/3))
f5_ax2.plot(botqpc[0,start:stop], reftauinfbot[start:stop], 'm--', label='Eq. 9, $V_{dc} = %1.1f \mu V$'%(idc[-6,0]*25813*1e6/3))

f5_ax2.annotate("", xytext=(-3.04, 0.6), xy=(-3.04, 0.9),
            arrowprops=dict(arrowstyle="->"))

f5_ax2.grid(ls='--', lw=0.4)
f5_ax2.set_ylabel('$\u03C4_{1}$')
f5_ax2.set_xlabel('$V_g$')
f5_ax2.legend(loc='lower right')
f5_ax2.annotate("", xytext=(-3.005, 0.4), xy=(-3.005, 0.8),
            arrowprops=dict(arrowstyle="->"))

###############
dat = load2dFig4(796, 151)
topqpc = dat['1']
idc = dat['2']
vr = dat['3']*100/q2
vt = dat['5']*100/s1
vxx = dat['7']
vrdc = dat['9']
vtdc = dat['10']
vsdc = vrdc + vtdc

G = 3*vt/(vt+vr)
tau = (1/G - 1)**-1

tauVfig5top = np.zeros(tau.shape)
begin=5
end=-5
# tauV = KNtheory(tau, Z, Ec, T, V)
poptfig5top = np.zeros((3,151))
pcovfig5top = np.zeros((3,151))
# tauinf = np.zeros((idc.shape[1]))
Z = 1
# T = 2*1*4.3e-6
# Ec = 12*T
# V = np.linspace(-3*Ec, 3*Ec,3001)
rreftau0top =  np.zeros((topqpc.shape[1]))
reftauinftop =  np.zeros((topqpc.shape[1]))
# tauV2 = KNtheory(tau, Z, Ec, T, V)
# start2 = 5
j=0
paramBounds = ([2e-6, 90e-6, 0.99999*Z], [8e-6, 110e-6, 1.000001*Z])
for i in range(idc.shape[1]):
    tau0 = tau[76,i]
    reftau0top[i] = tau0
    if i>=12 and i<46:

        j = i-10
#         print(botqpc[0,i])
#         ax[1].plot(0,0)
        f5_ax3.plot(1e6*(idc[begin:end,i] - idc[76,i])*25813/3, tau[begin:end,i],'k-', markersize=1, lw=1)
        # ax[1].plot(1e6*(vsdc[begin:end,i] - vsdc[75,i]), tau[begin:end,i],'r-', markersize=1, lw=1)
        lambdaKN = lambda tau0, T, Ec, Z : KNtheory(tau0, T, Ec, Z, idcstart=idc[50,0], idcstop=idc[100,0], npoints=51)
        poptfig5top[:,j], pcovifig5top = curve_fit(lambdaKN, tau0, tau[50:101,i], p0=[4.3e-6, 100e-6, Z], bounds=paramBounds)
        pcovfig5top[:,j] = np.diag(pcovifig5top)
        print(poptfig5top[:,j])
        tauVfig5top[begin:end,i] = KNtheoryfull(tau0, (idc[begin:end,i] - 1*idc[75,i])*25813/3, poptfig5top[0,j], poptfig5top[1,j], poptfig5top[2,j])
        print(tauVfig5top[end-1,i])
        reftauinftop[i] = tauVfig5top[end-1,i]
        f5_ax3.plot(1e6*(idc[begin:end,i] - 1*idc[75,i])*25813/3, tauVfig5top[begin:end,i], ls='--', color='m', lw=0.75)
        j=j+1


# ax[1].axvline(1e6*idc[41,0]*25813/3)
# ax[1].set_xlim(-100, 100)
f5_ax3.set_ylabel('V$_{dc}$  ($\mu$V)')
f5_ax3.set_xlabel('V$_{dc}$  ($\mu$V)')

f5_ax3.grid(ls='--', lw=0.4)
f5_ax3.axvline(idc[50,0]*25813/3*1e6, color='gray', ls='--')
f5_ax3.axvline(idc[100,0]*25813/3*1e6, color='gray', ls='--')

##############

start = 12
stop = 45

f5_ax4.plot(topqpc[0,start:stop], reftau0top[start:stop], 'k', label='$V_{dc} = %1.1f \mu V$' %(idc[75,0]*25813*1e6/3))
f5_ax4.plot(topqpc[0,start:stop], tau[-6,start:stop], 'b', label='$V_{dc} = %1.1f \mu V$' %(idc[-6,0]*25813*1e6/3))
f5_ax4.plot(topqpc[0,start:stop], reftauinftop[start:stop], 'm--',  label='Eq. 9, $V_{dc} = %1.1f \mu V$'%(idc[-6,0]*25813*1e6/3))

f5_ax4.annotate("", xytext=(-3.04, 0.6), xy=(-3.04, 0.9),
            arrowprops=dict(arrowstyle="->"))

f5_ax4.legend(loc='lower right')
f5_ax4.grid(ls='--', lw=0.4)
f5_ax4.set_ylabel('$\u03C4_{2}$')
f5_ax4.set_xlabel('$V_g$')
f5_ax4.annotate("", xytext=(-2.99, 0.2), xy=(-2.99, 0.75),
            arrowprops=dict(arrowstyle="->"))

f5_ax3.set_ylabel('$\\tilde{\u03C4}_2(V_{dc})$')
f5_ax1.set_ylabel('$\\tilde{\u03C4}_1(V_{dc})$')
f5_ax2.set_ylabel('$\\tilde{\u03C4}_2(V_{dc})$')
f5_ax4.set_ylabel('$\\tilde{\u03C4}_1(V_{dc})$')

fig5.savefig('DeviceB_DCB.pdf', dpi=300)
# fig5.savefig('DeviceB_DCB.eps', dpi=300)
# fig2.set_constrained_layout(False)
# fig3.tight_layout()

# fig1.canvas.draw()
# # we want the legend included in the bbox_inches='tight' calcs.
# # cbar1.set_in_layout(True)
# # cbar2.set_in_layout(True)


# # we don't want the layout to change at this point.
#fig1.tight_layout()

# skunk.display(svg)
# cairosvg.svg2pdf(bytestring=svg, write_to='/Users/praveen/Library/CloudStorage/GoogleDrive-prvn@stanford.edu/Shared drives/GGG GDrive/QDots-2/Papers/Hybrid Dot APL/figure5_Dec2025.pdf')
# fig1.savefig('Figure1.pdf', dpi=300)
# ax[0,2].axis('off')
# ax[1,2].axis('off')
# plt.tight_layout()

# plt.savefig("Figure4.eps")